# 05 — Silver Transactions

**Purpose:** Clean card transaction records from the existing Bronze table.

Steps:
1. Read Bronze transactions.
2. Cast all columns to the required data types.
3. Check missing or invalid values, including amounts.
4. Validate customer, merchant and card references.
5. Check card-to-customer ownership.
6. Remove duplicate transaction IDs.
7. Save Silver. Create Quarantine only when rejected rows exist.

**Amount rule used here:** amount must be successfully cast to decimal and be greater than zero. The original amount is retained in quarantine for inspection.

This notebook does not modify Bronze.

In [0]:
from pyspark.sql.functions import col, trim, upper, when, lit

BRONZE = "capstone_team01.bronze_schema"
SILVER = "capstone_team01.silver_schema"

spark.sql("CREATE SCHEMA IF NOT EXISTS capstone_team01.silver_schema")

## 1. Read Bronze transactions

In [0]:
transactions = spark.table(f"{BRONZE}.bronze_card_transactions")

print("Bronze transaction count:", transactions.count())
transactions.printSchema()
display(transactions.limit(5))

## 2. Cast columns

In [0]:
transactions = (transactions
    .withColumn("transaction_id", trim(col("transaction_id").cast("string")))
    .withColumn("customer_id", trim(col("customer_id").cast("string")))
    .withColumn("merchant_id", trim(col("merchant_id").cast("string")))
    .withColumn("card_id", trim(col("card_id").cast("string")))
    .withColumn("transaction_ts", col("transaction_ts").cast("timestamp"))
    .withColumn("amount", col("amount").cast("decimal(18,2)"))
    .withColumn("currency", upper(trim(col("currency").cast("string"))))
    .withColumn("channel", upper(trim(col("channel").cast("string"))))
    .withColumn("country", upper(trim(col("country").cast("string"))))
    .withColumn("authorization_code", trim(col("authorization_code").cast("string")))
    .withColumn("payment_status", upper(trim(col("payment_status").cast("string"))))
    .withColumn("device_id", trim(col("device_id").cast("string")))
)

transactions.printSchema()
display(transactions.limit(5))

# Inspect amounts that cannot be converted or are zero/negative
display(
    transactions.filter(
        col("amount").isNull() | (col("amount") <= 0)
    ).select("transaction_id", "amount")
)

## 3. Check reference records

In [0]:
# Read the master data from Silver
customers = spark.table(f"{SILVER}.silver_customer_master").select("customer_id").dropDuplicates()
merchants = spark.table(f"{SILVER}.silver_merchant_master").select("merchant_id").dropDuplicates()
cards = spark.table(f"{SILVER}.silver_card_master").select(
    "card_id",
    col("customer_id").alias("card_customer_id")
).dropDuplicates(["card_id"])

# Check whether each reference exists
transactions = transactions.join(
    customers.withColumn("customer_exists", lit(True)),
    "customer_id", "left"
)

transactions = transactions.join(
    merchants.withColumn("merchant_exists", lit(True)),
    "merchant_id", "left"
)

transactions = transactions.join(cards, "card_id", "left")

display(transactions.limit(5))

## 4. Separate valid and invalid transactions

In [0]:
# Invalid rows: missing required IDs, invalid timestamp or amount,
# unknown references, or card/customer ownership mismatch
transactions_invalid = transactions.filter(
    col("transaction_id").isNull() |
    (col("transaction_id") == "") |
    col("customer_id").isNull() |
    col("merchant_id").isNull() |
    col("card_id").isNull() |
    col("transaction_ts").isNull() |
    col("amount").isNull() |
    (col("amount") <= 0) |
    col("customer_exists").isNull() |
    col("merchant_exists").isNull() |
    col("card_customer_id").isNull() |
    (col("customer_id") != col("card_customer_id"))
)

# Valid rows
transactions_valid = transactions.filter(
    col("transaction_id").isNotNull() &
    (col("transaction_id") != "") &
    col("customer_id").isNotNull() &
    col("merchant_id").isNotNull() &
    col("card_id").isNotNull() &
    col("transaction_ts").isNotNull() &
    col("amount").isNotNull() &
    (col("amount") > 0) &
    col("customer_exists").isNotNull() &
    col("merchant_exists").isNotNull() &
    col("card_customer_id").isNotNull() &
    (col("customer_id") == col("card_customer_id"))
)

# Remove temporary reference-check columns
transactions_invalid = transactions_invalid.drop(
    "customer_exists", "merchant_exists", "card_customer_id"
)
transactions_valid = transactions_valid.drop(
    "customer_exists", "merchant_exists", "card_customer_id", "_amount_raw"
)

transactions_invalid = transactions_invalid.withColumn(
    "dq_reason",
    lit("Missing/invalid required field, amount <= 0, unknown reference or card ownership mismatch")
)

print("Invalid transaction rows:", transactions_invalid.count())
print("Valid transaction rows before duplicate check:", transactions_valid.count())

## 5. Handle duplicate transaction IDs

In [0]:
# Keep one row for each transaction_id
transactions_silver = transactions_valid.dropDuplicates(["transaction_id"])

# Capture the extra duplicate rows
transactions_duplicates = transactions_valid.exceptAll(transactions_silver)

transactions_duplicates = transactions_duplicates.withColumn(
    "dq_reason", lit("Duplicate transaction_id")
)

# Combine invalid rows and duplicate rows
transactions_quarantine = transactions_invalid.unionByName(transactions_duplicates, allowMissingColumns=True)

print("Silver transaction rows:", transactions_silver.count())
print("Quarantine transaction rows:", transactions_quarantine.count())

## 6. Save Silver and Quarantine

In [0]:
# Save the Silver table
(transactions_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.silver_card_transactions")
)

# Create Quarantine only when rejected rows exist
if transactions_quarantine.count() > 0:
    (transactions_quarantine.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER}.quarantine_card_transactions")
    )
    print("Quarantine table created.")
else:
    spark.sql(f"DROP TABLE IF EXISTS {SILVER}.quarantine_card_transactions")
    print("No rejected rows. Quarantine table was not created.")

print("Silver table saved.")

## 7. Final validation

In [0]:
# Compare source, Silver and Quarantine counts
bronze_count = spark.table(f"{BRONZE}.bronze_card_transactions").count()
silver_count = spark.table(f"{SILVER}.silver_card_transactions").count()

print("Bronze count:", bronze_count)
print("Silver count:", silver_count)

if spark.catalog.tableExists(f"{SILVER}.quarantine_card_transactions"):
    quarantine_count = spark.table(f"{SILVER}.quarantine_card_transactions").count()
else:
    quarantine_count = 0

print("Quarantine count:", quarantine_count)
print("Reconciled:", bronze_count == silver_count + quarantine_count)

# Check duplicate transaction IDs in Silver
display(
    spark.table(f"{SILVER}.silver_card_transactions")
    .groupBy("transaction_id")
    .count()
    .filter(col("count") > 1)
)

In [0]:
# Display transaction amount issues and missing optional attributes

transactions_review = (
    transactions
    .withColumn(
        "amount_issue",
        when(col("amount") == -999.00, "INVALID_AMOUNT")
        .when(col("amount") == 999999.99, "SUSPICIOUS_HIGH_AMOUNT")
        .otherwise("VALID_AMOUNT")
    )
    .withColumn(
        "authorization_code_missing",
        col("authorization_code").isNull() |
        (col("authorization_code") == "")
    )
    .withColumn(
        "device_id_missing",
        col("device_id").isNull() |
        (col("device_id") == "")
    )
)

# Display only records that need attention
display(
    transactions_review.filter(
        (col("amount_issue") != "VALID_AMOUNT") |
        col("authorization_code_missing") |
        col("device_id_missing")
    ).select(
        "transaction_id",
        "amount",
        "amount_issue",
        "authorization_code_missing",
        "device_id_missing"
    )
)